# 03 — EDA & Analysis

| Section | Content |
|---------|--------|
| A | HR distributions — histograms, boxplot by weekday, heatmap |
| B | Time-series grid per participant with anomaly markers |
| C | Anomaly detection summary (z-score + IQR) |
| D | Comparative analysis — participants × 8-day trend |
| E | HTML report export |

In [ ]:
import sys, logging, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.eda_utils import (
    plot_hr_distributions,
    plot_hr_boxplot_weekday,
    plot_hr_heatmap,
    plot_timeseries_grid,
    anomaly_summary_table,
    plot_participant_comparison,
    plot_weekly_trend,
    export_html_report,
)

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR   = REPO_ROOT / 'data' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
clean_path = PROCESSED_DIR / 'master_long_clean.parquet'
short_path = PROCESSED_DIR / 'master_short.parquet'

for p in (clean_path, short_path):
    if not p.exists():
        raise FileNotFoundError(f'{p} not found. Run notebooks 01 and 02 first.')

df_long  = pd.read_parquet(clean_path, engine='pyarrow')
df_short = pd.read_parquet(short_path,  engine='pyarrow')

participants = df_long['User_ID'].unique()
print(f'Long  : {df_long.shape}  |  Short: {df_short.shape}')
print(f'Participants: {participants}')

## Section A — Distributions

In [ ]:
fig_hist = plot_hr_distributions(df_long)
plt.show()

In [ ]:
fig_box = plot_hr_boxplot_weekday(df_long)
plt.show()

In [ ]:
fig_heat = plot_hr_heatmap(df_short)
plt.show()

## Section B — Time-Series per Participant

In [ ]:
ts_figures = {}
for pid in participants:
    fig = plot_timeseries_grid(df_long, participant=pid)
    ts_figures[pid] = fig
    plt.show()

## Section C — Anomaly Detection

In [ ]:
anom_table = anomaly_summary_table(df_long)
print('=== Anomaly summary per participant × day ===')
print(anom_table.to_string(index=False))

In [ ]:
# Per-participant roll-up
if not anom_table.empty and 'User_ID' in anom_table.columns:
    rollup = (
        anom_table
        .groupby('User_ID')
        .agg(total_rows=('total_rows','sum'),
             anomalies=('anomaly_count','sum'),
             pct_valid=('pct_valid','mean'))
        .reset_index()
    )
    rollup['anomaly_pct'] = (100 * rollup['anomalies'] / rollup['total_rows']).round(2)
    print(rollup.to_string(index=False))

## Section D — Comparative Analysis

In [ ]:
fig_cmp = plot_participant_comparison(df_short)
plt.show()

In [ ]:
fig_trend = plot_weekly_trend(df_short)
plt.show()

In [ ]:
# Descriptive statistics table
desc_cols = [c for c in ('FC_mean','FC_std','FC_min','FC_max','FC_range',
                          'duration_s','data_quality_pct') if c in df_short.columns]

if 'User_ID' in df_short.columns:
    stats_table = (
        df_short.groupby('User_ID')[desc_cols]
        .describe()
        .T
    )
else:
    stats_table = df_short[desc_cols].describe().T

stats_table

## Section E — Export HTML Report

In [ ]:
# Re-build figures for the report (avoids using already-shown/closed figures)
report_figures = {
    'A — HR Distributions per Participant' : plot_hr_distributions(df_long),
    'A — HR by Day of Week'                : plot_hr_boxplot_weekday(df_long),
    'A — Mean HR Heatmap'                  : plot_hr_heatmap(df_short),
    'B — Time Series'                      : [plot_timeseries_grid(df_long, p) for p in participants],
    'D — Participant Comparison'           : plot_participant_comparison(df_short),
    'D — Weekly Trend'                     : plot_weekly_trend(df_short),
}

flat_stats = df_short[desc_cols].describe().T.reset_index().rename(columns={'index': 'metric'})

meta_info = {
    'n_participants': int(df_long['User_ID'].nunique()),
    'n_days'        : int(df_long.groupby('User_ID')['Date'].nunique().sum()),
    'n_records'     : len(df_long),
}

report_path = export_html_report(
    figures=report_figures,
    stats=flat_stats,
    output_path=REPORTS_DIR / 'eda_report.html',
    meta=meta_info,
)

print(f'Report saved → {report_path.relative_to(REPO_ROOT)}')